# CVXOPT

**Seong Cho** · ACME Volume 2, Lab 15

**Lab objective:** introduce some of the basic optimization functions available in the CVXOPT package, for linear and quadratic programming.

<sub>Lab specification from the *Volume 2* lab manual of [Foundations of Applied Mathematics](https://foundations-of-applied-mathematics.github.io/) (BYU ACME), © Dr. J. Humpherys, used under [CC BY 3.0 US](http://creativecommons.org/licenses/by/3.0/us/). Problem statements are quoted from the spec; the notes are condensed from it. Solutions are my own.</sub>

> **Note.** This notebook requires [CVXOPT](http://cvxopt.org/) (`pip install cvxopt`). It was run with cvxopt 1.3.3.

> **Achtung!** CVXOPT is not part of the standard library, and it is only included in the Anaconda distribution for Python 3.6 for Linux and Mac. We recommend avoiding Windows machines for this lab. To install CVXOPT, use `conda install cvxopt` or `pip install cvxopt`.

The solution code uses `np.row_stack()`, which NumPy 2.0 deprecated as an alias of `np.vstack()`. The code is kept as written, and the setup cell hides that one `DeprecationWarning`.

In [1]:
import warnings
import numpy as np
import cvxopt as cvx
from cvxopt import matrix, solvers
from scipy.optimize import linprog

warnings.filterwarnings("ignore", message="`row_stack` alias is deprecated", category=DeprecationWarning)

## Linear Programs

CVXOPT is a package of Python functions and classes designed for convex optimization; this lab focuses on linear and quadratic programming. A *linear program* is a linear constrained optimization problem. CVXOPT uses the form

$$\begin{aligned} \text{minimize} \quad & \mathbf{c}^T\mathbf{x} \\ \text{subject to} \quad & G\mathbf{x} + \mathbf{s} = \mathbf{h} \\ & A\mathbf{x} = \mathbf{b} \\ & \mathbf{s} \succeq \mathbf{0}, \end{aligned}$$

where $A$ must have full row rank and the block matrix $\begin{bmatrix} G & A \end{bmatrix}^T$ full column rank. The *slack variable* $\mathbf{s}$ is not part of the objective; since $\mathbf{s} \succeq \mathbf{0}$, the constraint $G\mathbf{x} + \mathbf{s} = \mathbf{h}$ is equivalent to $G\mathbf{x} \preceq \mathbf{h}$.

For example, consider

$$\begin{aligned} \text{minimize} \quad & -4x_1 - 5x_2 \\ \text{subject to} \quad & x_1 + 2x_2 \le 3 \\ & 2x_1 + x_2 \le 3 \\ & x_1, x_2 \ge 0. \end{aligned}$$

The last two constraints become $-x_1, -x_2 \le 0$, so

$$G = \begin{bmatrix} 1 & 2 \\ 2 & 1 \\ -1 & 0 \\ 0 & -1 \end{bmatrix}, \qquad \mathbf{h} = \begin{bmatrix} 3 \\ 3 \\ 0 \\ 0 \end{bmatrix},$$

and the constraints are $G\mathbf{x} \preceq \mathbf{h}$. There are no equality constraints $A\mathbf{x} = \mathbf{b}$ here. CVXOPT uses its own matrix data type, which differs from the NumPy array especially in initialization:

In [2]:
c = matrix([-4., -5.])
G = matrix([[1., 2., -1., 0.],[2., 1., 0., -1.]])
h = matrix([ 3., 3., 0., 0.])
print(G)

[ 1.00e+00  2.00e+00]
[ 2.00e+00  1.00e+00]
[-1.00e+00  0.00e+00]
[ 0.00e+00 -1.00e+00]



> **Achtung!** CVXOPT matrices are initialized column-wise rather than row-wise (as in the case of NumPy).

Alternatively, initialize the arrays in NumPy and convert them. The lab initializes non-trivial matrices this way for consistency. Be sure the entries are floats!

In [3]:
c = np.array([-4., -5.])
G = np.array([[1., 2.],[2., 1.],[-1., 0.],[0., -1]])
h = np.array([3., 3., 0., 0.])

# Convert the arrays to the CVXOPT matrix type.
c = matrix(c)
G = matrix(G)
h = matrix(h)

The linear programming solver is `solvers.lp()`; pass it $\mathbf{c}$, $G$, and $\mathbf{h}$. By default it prints the progress of each iteration:

In [4]:
sol = solvers.lp(c, G, h)
print(sol['x'])
print(sol['primal objective'])
print(type(sol['x']))

     pcost       dcost       gap    pres   dres   k/t
 0: -8.1000e+00 -1.8300e+01  4e+00  0e+00  8e-01  1e+00
 1: -8.8055e+00 -9.4357e+00  2e-01  2e-16  4e-02  3e-02
 2: -8.9981e+00 -9.0049e+00  2e-03  2e-16  5e-04  4e-04
 3: -9.0000e+00 -9.0000e+00  2e-05  2e-16  5e-06  4e-06
 4: -9.0000e+00 -9.0000e+00  2e-07  1e-16  5e-08  4e-08
Optimal solution found.
[ 1.00e+00]
[ 1.00e+00]

-8.999999811406719
<class 'cvxopt.base.matrix'>


The spec, written for Python 2, shows the type as `<type 'cvxopt.base.matrix'>`; Python 3 prints `<class ...>`.

> **Note.** Although it is often helpful to see the progress of each iteration of the algorithm, you may suppress this output by first running `solvers.options['show_progress'] = False`.

The rest of the notebook does this, except where the spec shows the progress output.

In [5]:
solvers.options['show_progress'] = False

`solvers.lp()` returns a dictionary of useful information; this lab uses only `'x'` and the `'primal objective'` (the minimum value of the objective function).

> **Achtung!** Note that the minimizer of the `solvers.lp()` function returns a `cvxopt.base.matrix` object. In order to use the minimizer again in other algebraic expressions, you need to convert it first to a flattened numpy array, which can be done quickly with `np.ravel()`. Please return all minimizers in this lab as flattened numpy arrays.

## Problem 1: A linear program

> **Problem 1.** Solve the following convex optimization problem:
>
> $$\begin{aligned} \text{minimize} \quad & 2x_1 + x_2 + 3x_3 \\ \text{subject to} \quad & x_1 + 2x_2 \ge 3 \\ & 2x_1 + 10x_2 + 3x_3 \ge 10 \\ & x_1 \ge 0 \\ & x_2 \ge 0 \\ & x_3 \ge 0 \end{aligned}$$
>
> Report the values for $\mathbf{x}$ and the objective value that you obtain. Remember to make the necessary adjustments so that all inequality constraints are $\le$ rather than $\ge$.

In [6]:
def prob1():
    """Solve the following convex optimization problem:

    minimize        2x + y + 3z
    subject to      x + 2y          >= 3
                    2x + 10y + 3z   >= 10
                    x               >= 0
                    y               >= 0
                    z               >= 0

    Returns (in order):
        The optimizer x (ndarray)
        The optimal value (sol['primal objective'])
    """
    c = cvx.matrix([2.,1.,3.])
    G = cvx.matrix([[-1.,-2.,-1.,0.,0.],[-2.,-10.,0.,-1.,0.],[0.,-3.,0.,0.,-1.]])
    h = cvx.matrix([-3.,-10.,0.,0.,0.])
    sol = cvx.solvers.lp(c, G, h)
    return np.ravel(sol['x']), sol['primal objective']

Checked against `scipy.optimize.linprog()` (whose variables are nonnegative by default):

In [7]:
x, val = prob1()
print("x =", np.round(x, 6), "  objective =", val)
ref = linprog([2, 1, 3], A_ub=[[-1, -2, 0], [-2, -10, -3]], b_ub=[-3, -10])
print("linprog: x =", ref.x, "  objective =", ref.fun)
assert np.allclose(x, ref.x, atol=1e-6) and np.isclose(val, ref.fun)

x = [-0.   1.5  0. ]   objective = 1.4999999996249473
linprog: x = [0.  1.5 0. ]   objective = 1.5


## $l_1$ Minimization

The $l_1$ minimization problem is to

$$\begin{aligned} \text{minimize} \quad & \|\mathbf{x}\|_1 \\ \text{subject to} \quad & A\mathbf{x} = \mathbf{b}. \end{aligned}$$

It becomes a linear program by introducing an additional vector $\mathbf{u}$ of length $n$ and solving

$$\begin{aligned} \text{minimize} \quad & \begin{bmatrix} \mathbf{1}^T & \mathbf{0}^T \end{bmatrix} \begin{bmatrix} \mathbf{u} \\ \mathbf{x} \end{bmatrix} \\ \text{subject to} \quad & \begin{bmatrix} -I & I \\ -I & -I \end{bmatrix} \begin{bmatrix} \mathbf{u} \\ \mathbf{x} \end{bmatrix} \preceq \begin{bmatrix} \mathbf{0} \\ \mathbf{0} \end{bmatrix}, \\ & \begin{bmatrix} 0 & A \end{bmatrix} \begin{bmatrix} \mathbf{u} \\ \mathbf{x} \end{bmatrix} = \mathbf{b}. \end{aligned}$$

This gives the optimal $\mathbf{u}$ and $\mathbf{x}$, but only $\mathbf{x}$ is needed.

## Problem 2: $l_1$ minimization

> **Problem 2.** Write a function called `l1Min()` that takes a matrix $A$ and vector $\mathbf{b}$ as inputs, and solves the $l_1$ optimization problem. Report the values for $\mathbf{x}$ and the objective value. Remember to first discard the unnecessary $\mathbf{u}$ values from the minimizer.

The function prints the arrays it builds (the grader asked for these print statements to be commented out; they are kept as written).

In [8]:
# Problem 2
def l1Min(A, b):
    """Calculate the solution to the optimization problem

        minimize    ||x||_1
        subject to  Ax = b

    Parameters:
        A ((m,n) ndarray)
        b ((m, ) ndarray)

    Returns:
        The optimizer x (ndarray), without any slack variables u
        The optimal value (sol['primal objective'])
    """
    m = len(A)
    n = len(A[0])
    c = cvx.matrix(np.concatenate([np.ones(n), np.zeros(n)]))
    iden = np.eye(n)
    r1 = np.column_stack([-iden, iden])
    r2 = np.column_stack([-iden, -iden])
    G = cvx.matrix(np.row_stack([r1, r2]))
    h = cvx.matrix(np.zeros(2*n))
    Z = np.zeros((m,n))
    Am = cvx.matrix(np.column_stack([Z, A]))
    print(c)
    print(G)
    print(h)
    print(Am)
    print(b)
    sol = cvx.solvers.lp(c,G,h,Am,cvx.matrix(b))
    return np.ravel(sol['x'][n:]), sol['primal objective']

A small underdetermined test system:

$$A = \begin{bmatrix} 1 & 2 & 1 & 1 \\ 0 & 3 & -2 & -1 \end{bmatrix}, \qquad \mathbf{b} = \begin{bmatrix} 7 \\ 4 \end{bmatrix}.$$

In [9]:
A = np.array([[1., 2, 1, 1], [0, 3, -2, -1]])
b = np.array([7., 4])
x1, val1 = l1Min(A, b)

[ 1.00e+00]
[ 1.00e+00]
[ 1.00e+00]
[ 1.00e+00]
[ 0.00e+00]
[ 0.00e+00]
[ 0.00e+00]
[ 0.00e+00]

[-1.00e+00 -0.00e+00 -0.00e+00 -0.00e+00  1.00e+00  0.00e+00  0.00e+00 ... ]
[-0.00e+00 -1.00e+00 -0.00e+00 -0.00e+00  0.00e+00  1.00e+00  0.00e+00 ... ]
[-0.00e+00 -0.00e+00 -1.00e+00 -0.00e+00  0.00e+00  0.00e+00  1.00e+00 ... ]
[-0.00e+00 -0.00e+00 -0.00e+00 -1.00e+00  0.00e+00  0.00e+00  0.00e+00 ... ]
[-1.00e+00 -0.00e+00 -0.00e+00 -0.00e+00 -1.00e+00 -0.00e+00 -0.00e+00 ... ]
[-0.00e+00 -1.00e+00 -0.00e+00 -0.00e+00 -0.00e+00 -1.00e+00 -0.00e+00 ... ]
[-0.00e+00 -0.00e+00 -1.00e+00 -0.00e+00 -0.00e+00 -0.00e+00 -1.00e+00 ... ]
[-0.00e+00 -0.00e+00 -0.00e+00 -1.00e+00 -0.00e+00 -0.00e+00 -0.00e+00 ... ]

[ 0.00e+00]
[ 0.00e+00]
[ 0.00e+00]
[ 0.00e+00]
[ 0.00e+00]
[ 0.00e+00]
[ 0.00e+00]
[ 0.00e+00]

[ 0.00e+00  0.00e+00  0.00e+00  0.00e+00  1.00e+00  2.00e+00  1.00e+00 ... ]
[ 0.00e+00  0.00e+00  0.00e+00  0.00e+00  0.00e+00  3.00e+00 -2.00e+00 ... ]

[7. 4.]


For the check, `linprog()` solves the same problem with the split $\mathbf{x} = \mathbf{x}^+ - \mathbf{x}^-$, $\mathbf{x}^\pm \succeq \mathbf{0}$, minimizing $\mathbf{1}^T(\mathbf{x}^+ + \mathbf{x}^-)$. The $l_1$ minimizer need not be unique, so the check compares the optimal values and confirms that `l1Min()`'s point is feasible and attains that value.

In [10]:
print("x =", np.round(x1, 6), "  objective =", val1)
n = A.shape[1]
ref = linprog(np.ones(2*n), A_eq=np.hstack([A, -A]), b_eq=b)
print("linprog: x =", np.round(ref.x[:n] - ref.x[n:], 6), "  objective =", ref.fun)
assert np.isclose(val1, ref.fun) and np.isclose(np.abs(x1).sum(), ref.fun, atol=1e-6)
assert np.allclose(A @ x1, b)

x = [0.       2.571429 1.857143 0.      ]   objective = 4.428571420259736
linprog: x = [0.       2.571429 1.857143 0.      ]   objective = 4.428571428571429


## The Transportation Problem

A piano company needs to transport thirteen pianos from three supply centers (1, 2, 3) to two demand centers (4, 5), minimizing shipping costs while meeting the demand. How many pianos should each supply center send to each demand center?

| Supply center | Pianos available |
|---|---|
| 1 | 7 |
| 2 | 2 |
| 3 | 4 |

*Table 15.1: Number of pianos available at each supply center.*

| Demand center | Pianos needed |
|---|---|
| 4 | 5 |
| 5 | 8 |

*Table 15.2: Number of pianos needed at each demand center.*

| Supply center | Demand center | Cost of transportation | Number of pianos |
|---|---|---|---|
| 1 | 4 | 4 | $p$ |
| 1 | 5 | 7 | $q$ |
| 2 | 4 | 6 | $r$ |
| 2 | 5 | 8 | $s$ |
| 3 | 4 | 8 | $t$ |
| 3 | 5 | 9 | $u$ |

*Table 15.3: Cost of transporting one piano from a supply center to a demand center.*

The nonnegative variables $p, q, r, s, t, u$ satisfy three supply constraints and two demand constraints,

$$p + q = 7, \quad r + s = 2, \quad t + u = 4, \quad p + r + t = 5, \quad q + s + u = 8,$$

and the objective is $4p + 7q + 6r + 8s + 8t + 9u$. We want integer answers, and that added constraint in general turns out to be an NP-hard problem (the field of *integer linear programming*, beyond the scope of this lab). Fortunately, this particular problem can be treated as a standard linear program and still give integer solutions.

Here $G = -I_6$ and $\mathbf{h} = \mathbf{0}$ constrain the variables to be nonnegative, and $A$, $\mathbf{b}$ hold the equality constraints. The spec's first attempt, with the progress output it shows:

In [11]:
c = matrix([4., 7., 6., 8., 8., 9])
G = matrix(-1*np.eye(6))
h = matrix(np.zeros(6))
A = matrix(np.array([[1.,1.,0.,0.,0.,0.],
                     [0.,0.,1.,1.,0.,0.],
                     [0.,0.,0.,0.,1.,1.],
                     [1.,0.,1.,0.,1.,0.],
                     [0.,1.,0.,1.,0.,1.]]))
b = matrix([7., 2., 4., 5., 8.])
sol = solvers.lp(c, G, h, A, b, options={'show_progress': True})
print(sol['x'])
print(sol['primal objective'])

     pcost       dcost       gap    pres   dres   k/t
 0:  8.9500e+01  8.9500e+01  2e+01  6e-16  2e-01  1e+00
 1:  8.7023e+01  8.7044e+01  3e+00  1e-15  3e-02  2e-01
 2:  8.6120e+01  8.6130e+01  6e-01  4e-15  6e-03  4e-02
Terminated (singular KKT matrix).
[ 4.94e+00]
[ 2.06e+00]
[ 3.56e-03]
[ 2.00e+00]
[ 5.81e-02]
[ 3.94e+00]

86.119723009483


The spec's run stopped after iteration 0 with the solution $(3, 4, .5, 1.5, 1.5, 2.5)$ and objective 89.5. CVXOPT 1.3.3 gets two iterations further before the same failure, so its numbers differ, but the problems are the same: the algorithm terminated prematurely (due to a singular matrix), and the solution does not consist of integer entries.

The cause: $A$ is required to have full row rank, but its rows are linearly dependent (the three supply rows and the two demand rows each add up to the same row of ones). The fix is to convert the last equality constraint into inequality constraints, so that the remaining equality constraints have linearly independent rows. An equality constraint such as $x + 2y - 3z = 4$ is equivalent to the pair $x + 2y - 3z \le 4$ and $x + 2y - 3z \ge 4$, that is,

$$x + 2y - 3z \le 4, \qquad -x - 2y + 3z \le -4.$$

Applying this to the last equality constraint gives a new $G$ with additional rows, a longer $\mathbf{h}$, and a smaller $A$ and $\mathbf{b}$.

## Problem 3: The transportation problem

> **Problem 3.** Solve the problem by converting the last equality constraint into an inequality constraint. Report the optimal values for $\mathbf{x}$ and the objective function.

In [12]:
# Problem 3
def prob3():
    """Solve the transportation problem by converting the last equality constraint
    into inequality constraints.

    Returns (in order):
        The optimizer x (ndarray)
        The optimal value (sol['primal objective'])
    """
    c = cvx.matrix([4.,7.,6.,8.,8.,9.])
    g1 = -1*np.eye(6)
    g2 = np.array([0.,1.,0.,1.,0.,1.])
    g = np.row_stack([g1, g2, -g2])
    G = cvx.matrix(g)
    h = cvx.matrix(np.concatenate([np.zeros(6), [8.,-8.]]))
    A = cvx.matrix(np.array([[1.,1.,0.,0.,0.,0.],
                             [0.,0.,1.,1.,0.,0.],
                             [0.,0.,0.,0.,1.,1.],
                             [1.,0.,1.,0.,1.,0.]]))
    b = cvx.matrix([7.,2.,4.,5.])
    sol = cvx.solvers.lp(c,G,h,A,b)
    return np.ravel(sol['x']), sol['primal objective']

The solution is integral: 5 pianos $1 \to 4$, 2 pianos $1 \to 5$, 2 pianos $2 \to 5$, 4 pianos $3 \to 5$, at cost 86. `linprog()`, which handles the dependent equality constraints, agrees.

In [13]:
x3, val3 = prob3()
print("(p, q, r, s, t, u) =", np.round(x3, 6), "  cost =", val3)
ref = linprog([4, 7, 6, 8, 8, 9],
              A_eq=[[1,1,0,0,0,0], [0,0,1,1,0,0], [0,0,0,0,1,1], [1,0,1,0,1,0], [0,1,0,1,0,1]],
              b_eq=[7, 2, 4, 5, 8])
print("linprog:", ref.x, "  cost =", ref.fun)
assert np.allclose(x3, ref.x, atol=1e-6) and np.isclose(val3, ref.fun)

(p, q, r, s, t, u) = [ 5.  2. -0.  2. -0.  4.]   cost = 85.99999998207572
linprog: [5. 2. 0. 2. 0. 4.]   cost = 86.0


## Quadratic Programming

Quadratic programming differs from linear programming in that the objective function is quadratic; the constraints, if any, have the same form, so $G$, $\mathbf{h}$, $A$, and $\mathbf{b}$ are optional. The formulation is

$$\begin{aligned} \text{minimize} \quad & \frac{1}{2}\mathbf{x}^TP\mathbf{x} + \mathbf{q}^T\mathbf{x} \\ \text{subject to} \quad & G\mathbf{x} \preceq \mathbf{h} \\ & A\mathbf{x} = \mathbf{b}, \end{aligned}$$

where $P$ is a positive semidefinite symmetric matrix. Again $A$ must have full row rank, and the block matrix $\begin{bmatrix} P & G & A \end{bmatrix}^T$ full column rank.

For example, take the unconstrained $f(y, z) = 2y^2 + 2yz + z^2 + y - z$. With

$$P = \begin{bmatrix} a & b \\ b & c \end{bmatrix}, \quad \mathbf{q} = \begin{bmatrix} d \\ e \end{bmatrix}, \quad \mathbf{x} = \begin{bmatrix} y \\ z \end{bmatrix},$$

$$\frac{1}{2}\mathbf{x}^TP\mathbf{x} + \mathbf{q}^T\mathbf{x} = \frac{1}{2}ay^2 + byz + \frac{1}{2}cz^2 + dy + ez,$$

so $a = 4$, $b = 2$, $c = 2$, $d = 1$, $e = -1$. The quadratic programming solver is `solvers.qp()`:

In [14]:
P = matrix(np.array([[4., 2.], [2., 2.]]))
q = matrix([1., -1.])
sol = solvers.qp(P, q)
print(sol['x'])
print(sol['primal objective'])

[-1.00e+00]
[ 1.50e+00]

-1.25


This matches the spec's output, $(-1, 1.5)$ and $-1.25$ (the spec's `print sol['primal objective']` is Python 2 syntax).

## Problem 4: A quadratic program

> **Problem 4.** Find the minimizer and minimum of
>
> $$g(x, y, z) = \frac{3}{2}x^2 + 2xy + xz + 2y^2 + 2yz + \frac{3}{2}z^2 + 3x + z$$

In [15]:
# Problem 4
def prob4():
    """Find the minimizer and minimum of

    g(x,y,z) = (3/2)x^2 + 2xy + xz + 2y^2 + 2yz + (3/2)z^2 + 3x + z

    Returns (in order):
        The optimizer x (ndarray)
        The optimal value (sol['primal objective'])
    """
    P = cvx.matrix(np.array([[3.,2.,1.],[2.,4.,2.],[1.,2.,3.]]))
    q = cvx.matrix([3.,0.,1.])
    sol = cvx.solvers.qp(P,q)
    return np.ravel(sol['x']), sol['primal objective']

Since $P$ is positive definite and there are no constraints, the minimizer solves $P\mathbf{x} = -\mathbf{q}$; evaluating $g$ there directly gives the minimum.

In [16]:
x4, val4 = prob4()
print("minimizer =", x4, "  minimum =", val4)

Pn = np.array([[3., 2, 1], [2, 4, 2], [1, 2, 3]])
qn = np.array([3., 0, 1])
x_ref = np.linalg.solve(Pn, -qn)
g = lambda x, y, z: 1.5*x**2 + 2*x*y + x*z + 2*y**2 + 2*y*z + 1.5*z**2 + 3*x + z
print("solve(P, -q) =", x_ref, "  g there =", g(*x_ref))
assert np.allclose(x4, x_ref) and np.isclose(val4, g(*x_ref))

minimizer = [-1.5  1.  -0.5]   minimum = -2.5
solve(P, -q) = [-1.5  1.  -0.5]   g there = -2.5


## Problem 5: $l_2$ minimization

> **Problem 5.** The $l_2$ minimization problem is to
>
> $$\begin{aligned} \text{minimize} \quad & \|\mathbf{x}\|_2 \\ \text{subject to} \quad & A\mathbf{x} = \mathbf{b}. \end{aligned}$$
>
> This problem is equivalent to a quadratic program, since $\|\mathbf{x}\|^2 = \mathbf{x}^T\mathbf{x}$. Write a function called `l2Min()` that takes a matrix $A$ and vector $\mathbf{b}$ as inputs and solves the $l_2$ minimization problem. Report the values for $\mathbf{x}$ and the objective value.

The function prints the shape of $A$ (kept as written). With $P = 2I$ the objective $\frac{1}{2}\mathbf{x}^TP\mathbf{x}$ is $\mathbf{x}^T\mathbf{x} = \|\mathbf{x}\|_2^2$, so the reported objective value is the squared norm.

In [17]:
# Problem 5
def l2Min(A, b):
    """Calculate the solution to the optimization problem

        minimize    ||x||_2
        subject to  Ax = b

    Parameters:
        A ((m,n) ndarray)
        b ((m, ) ndarray)

    Returns:
        The optimizer x (ndarray)
        The optimal value (sol['primal objective'])
    """
    m = len(A)
    n = len(A[0])
    print(m,n)
    P = cvx.matrix(2*np.eye(n))
    q = cvx.matrix(np.zeros(n))
    A = cvx.matrix(A)
    b = cvx.matrix(b)
    sol = cvx.solvers.qp(P,q, A=A, b=b)
    return np.ravel(sol['x']), sol['primal objective']

On the system from Problem 2, the minimum-norm solution of $A\mathbf{x} = \mathbf{b}$ is also given by the Moore-Penrose pseudoinverse, $\mathbf{x} = A^\dagger\mathbf{b}$:

In [18]:
A = np.array([[1., 2, 1, 1], [0, 3, -2, -1]])
b = np.array([7., 4])
x5, val5 = l2Min(A, b)
x_ref = np.linalg.pinv(A) @ b
print("x =", x5, "  objective =", val5)
print("pinv(A) @ b =", x_ref, "  ||x||^2 =", x_ref @ x_ref)
assert np.allclose(x5, x_ref) and np.isclose(val5, x_ref @ x_ref)

2 4
x = [0.96629213 2.16853933 0.80898876 0.88764045]   objective = 7.078651685393258
pinv(A) @ b = [0.96629213 2.16853933 0.80898876 0.88764045]   ||x||^2 = 7.0786516853932575


## Allocation Models

Allocation models lead to simple linear programs: they allocate a valuable resource among competing needs. This example is taken from "Optimization in Operations Research" by Ronald L. Rardin.

The U.S. Forest service has used an allocation model to manage national forests. The land is divided into *analysis areas*, and several land management policies (*prescriptions*) are proposed and evaluated for each area. An allocation is how much land (in acreage) in each analysis area is assigned to each prescription; we seek the best allocation subject to forest-wide restrictions on land use.

The file `ForestData.npy` contains data for a fictional national forest (Table 15.4 in the spec): 7 areas of analysis with 3 prescriptions each. Its columns are

1. $i$, the area of analysis (given on the first row of each area),
2. $s_i$, the size of the analysis area (in thousands of acres),
3. $j$, the prescription number,
4. $p_{i,j}$, the net present value (NPV) per acre in area $i$ under prescription $j$,
5. $t_{i,j}$, the protected timber yield per acre,
6. $g_{i,j}$, the protected grazing capability per acre,
7. $w_{i,j}$, the wilderness index rating (0 to 100).

In [19]:
np.set_printoptions(suppress=True)
print(np.load('ForestData.npy'))
np.set_printoptions(suppress=False)

[[  1.    75.     1.   503.   310.     0.01  40.  ]
 [  0.     0.     2.   140.    50.     0.04  80.  ]
 [  0.     0.     3.   203.     0.     0.    95.  ]
 [  2.    90.     1.   675.   198.     0.03  55.  ]
 [  0.     0.     2.   100.    46.     0.06  60.  ]
 [  0.     0.     3.    45.     0.     0.    65.  ]
 [  3.   140.     1.   630.   210.     0.04  45.  ]
 [  0.     0.     2.   105.    57.     0.07  55.  ]
 [  0.     0.     3.    40.     0.     0.    60.  ]
 [  4.    60.     1.   330.   112.     0.01  30.  ]
 [  0.     0.     2.    40.    30.     0.02  35.  ]
 [  0.     0.     3.   295.     0.     0.    90.  ]
 [  5.   212.     1.   105.    40.     0.05  60.  ]
 [  0.     0.     2.   460.    32.     0.08  60.  ]
 [  0.     0.     3.   120.     0.     0.    70.  ]
 [  6.    98.     1.   490.   105.     0.02  35.  ]
 [  0.     0.     2.    55.    25.     0.03  50.  ]
 [  0.     0.     3.   180.     0.     0.    75.  ]
 [  7.   113.     1.   705.   213.     0.02  40.  ]
 [  0.     0

Let $x_{i,j}$ be the amount of land in area $i$ allocated to prescription $j$; an allocation is the vector of the $7 \cdot 3 = 21$ values $x_{i,j}$. The goal is to maximize net present value while producing at least 40 million board-feet of timber, at least 5 thousand animal-unit months of grazing, and keeping the average wilderness index at least 70. The allocation is nonnegative, and all of the land must be allocated precisely. Since acres are in thousands, the timber and grazing constraints are divided by 1000 in the setup, and this is compensated for after the solution is obtained. In summary,

$$\begin{aligned} \text{maximize} \quad & \sum_{i=1}^7 \sum_{j=1}^3 p_{i,j}x_{i,j} \\ \text{subject to} \quad & \sum_{j=1}^3 x_{i,j} = s_i \quad \text{for } i = 1, \ldots, 7 \\ & \sum_{i=1}^7 \sum_{j=1}^3 t_{i,j}x_{i,j} \ge 40{,}000 \\ & \sum_{i=1}^7 \sum_{j=1}^3 g_{i,j}x_{i,j} \ge 5 \\ & \frac{1}{788}\sum_{i=1}^7 \sum_{j=1}^3 w_{i,j}x_{i,j} \ge 70 \\ & x_{i,j} \ge 0 \quad \text{for } i = 1, \ldots, 7 \text{ and } j = 1, 2, 3. \end{aligned}$$

(788 thousand acres is the total size of the forest.)

## Problem 6: Forest planning

> **Problem 6.** Solve the problem above. Return the minimizer $\mathbf{x}$ of $x_{i,j}$'s. Also return the maximum total net present value, which will be equal to the primal objective of the appropriately minimized linear function, multiplied by -1000. (This final multiplication after we have obtained a solution changes our answer to be a maximum, and compensates for the data being in thousands of acres).

`np.float` was removed in NumPy 1.24, so the three `.astype(np.float)` calls now use the built-in `float` (the same type).

In [20]:
# Problem 6
def prob6():
    """Solve the allocation model problem in 'ForestData.npy'.
    Note that the first three rows of the data correspond to the first
    analysis area, the second group of three rows correspond to the second
    analysis area, and so on.

    Returns (in order):
        The optimizer x (ndarray)
        The optimal value (sol['primal objective']*-1000)
    """
    data = np.load('ForestData.npy')
    s = data[:,1]
    b = cvx.matrix(s[::3].astype(float))       # Changed: np.float -> float (np.float was removed in NumPy 1.24).
    p = data[:,3]
    c = cvx.matrix(-p)
    t = data[:,4]
    g = data[:,5]
    w = data[:,6]

    G1 = np.row_stack([t,g,w])
    h1 = np.array([40000., 5., 55160.])
    G2 = np.eye(21).astype(float)               # Changed: np.float -> float (removed in NumPy 1.24).
    h2 = np.zeros(21).astype(float)             # Changed: np.float -> float (removed in NumPy 1.24).
    G = cvx.matrix(np.row_stack([-G1, -G2]))
    h = cvx.matrix(np.concatenate([-h1, -h2]))
    R = [np.concatenate([np.array([0]*3*i), np.ones(3), np.array([0]*3*(6-i))]) for i in range(7)]
    A = cvx.matrix(np.row_stack(R))
    sol = cvx.solvers.lp(c, G, h, A, b)
    return np.ravel(sol['x']), sol['primal objective']*-1000

The allocation, one row per analysis area, and the maximum NPV. The check builds the same linear program independently from the data file and solves it with `linprog()`; it also confirms every constraint.

In [21]:
x6, npv = prob6()
print(np.round(x6.reshape(7, 3), 4))
print(f"Maximum total NPV: {npv:,.0f}")

data = np.load('ForestData.npy')
p, t, g, w = data[:,3], data[:,4], data[:,5], data[:,6]
sizes = data[::3, 1]
A_eq = np.kron(np.eye(7), np.ones(3))                 # Each area's three allocations sum to s_i.
A_ub = -np.vstack([t, g, w/sizes.sum()])
b_ub = -np.array([40000, 5, 70])
ref = linprog(-p, A_ub=A_ub, b_ub=b_ub, A_eq=A_eq, b_eq=sizes)
print(f"linprog NPV:       {-ref.fun*1000:,.0f}")
assert np.isclose(npv, -ref.fun*1000) and np.allclose(x6, ref.x, atol=1e-4)
assert np.allclose(A_eq @ x6, sizes) and np.all(A_ub @ x6 <= b_ub + 1e-6) and np.all(x6 >= -1e-6)

[[  0.   0.  75.]
 [ 90.   0.   0.]
 [140.   0.   0.]
 [  0.   0.  60.]
 [  0. 154.  58.]
 [  0.   0.  98.]
 [  0.   0. 113.]]
Maximum total NPV: 322,514,999
linprog NPV:       322,515,000


You can learn more about CVXOPT at http://cvxopt.org/index.html.